In [ ]:
# ===== CELL 1: Setup =====
!pip install torchvision torch torchaudio
import torch
import torch.nn as nn
from torchvision.models import xception
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path
import cv2
import numpy as np
print("✅ Video Setup")


In [ ]:
# ===== CELL 2: VideoFrameDataset =====
class VideoFrameDataset(Dataset):
    def __init__(self, frames_root, transform=None, seq_len=32):
        self.root = Path(frames_root)
        self.transform = transform
        self.seq_len = seq_len
        self.video_list = []
        
        for video_dir in self.root.glob("*"):
            frames = sorted(list((video_dir / 'fake').glob("*.jpg")))  # or detect label
            if len(frames) >= self.seq_len:
                self.video_list.append((video_dir, 1))  # fake label
    
    def __getitem__(self, idx):
        video_dir, label = self.video_list[idx]
        frames = sorted(list((video_dir / 'fake').glob("*.jpg")))[:self.seq_len]
        
        frame_tensors = []
        for frame_path in frames:
            img = Image.open(frame_path).convert('RGB')
            if self.transform:
                frame_tensors.append(self.transform(img))
        
        # Pad if needed
        while len(frame_tensors) < self.seq_len:
            frame_tensors.append(frame_tensors[-1])
            
        return torch.stack(frame_tensors), torch.tensor(label)
    
    def __len__(self):
        return len(self.video_list)

video_transform = transforms.Compose([
    transforms.Resize((299, 299)),  # Xception input
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [ ]:
# ===== CELL 3: Xception + BiLSTM Model =====
class VideoModel(nn.Module):
    def __init__(self, seq_len=32):
        super().__init__()
        backbone = xception(pretrained=True)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # Remove FC
        self.lstm = nn.LSTM(2048, 512, bidirectional=True, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(1024, 2)
        self.seq_len = seq_len
    
    def forward(self, x):  # [B, T, C, H, W]
        B, T, C, H, W = x.shape
        features = []
        
        for t in range(T):
            feat = self.backbone(x[:, t]).squeeze(-1).squeeze(-1)  # [B, 2048]
            features.append(feat)
        
        seq_feat = torch.stack(features, dim=1)  # [B, T, 2048]
        lstm_out, _ = self.lstm(seq_feat)
        final_feat = lstm_out[:, -1]  # Last timestep
        return self.fc(final_feat)

model = VideoModel().cuda()
print(model)


In [ ]:
# ===== CELL 4: Training =====
train_ds = VideoFrameDataset('data/videos/frames', video_transform)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)  # Small batch for memory

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(3):
    total_loss = 0
    for videos, labels in train_loader:
        videos = videos.cuda()
        labels = labels.cuda()
        
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "models/video_xception_bilstm.pth")
print("✅ Video model trained")


In [ ]:
# ===== CELL 5: Test Inference =====
model.eval()
test_video_dir = next(Path("data/videos/frames").glob("*"))
frames = sorted(list((test_video_dir / 'fake').glob("*.jpg")))[:32]
frame_tensors = torch.stack([video_transform(Image.open(f).convert('RGB')) for f in frames]).unsqueeze(0).cuda()

with torch.no_grad():
    output = model(frame_tensors)
    prob = torch.softmax(output, dim=1)[0][1].item()
print(f"Sample video fake probability: {prob:.3f}")
